In [53]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

df = pd.read_csv('data/processed/tunizi_clean_binary.csv')

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), max_features=50000, min_df=2)
X_train_c = vec_char.fit_transform(X_train)
X_test_c = vec_char.transform(X_test)

model_char = LogisticRegression(max_iter=1000, random_state=42)
model_char.fit(X_train_c, y_train)

print(classification_report(y_test, model_char.predict(X_test_c)))



              precision    recall  f1-score   support

          -1       0.82      0.86      0.84      8754
           1       0.88      0.84      0.86     10542

    accuracy                           0.85     19296
   macro avg       0.85      0.85      0.85     19296
weighted avg       0.85      0.85      0.85     19296



In [54]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for t in ["nakrhek", "nkrahek", "nekrahek", "chokrane 3al khedma mte3kom", "nhebbou barcha"]:
    X = vec_char.transform([clean_text(t)])
    p = model_char.predict_proba(X)[0]
    print(f"{t} → {'Positif' if p[1] > 0.5 else 'Négatif'} (confiance {max(p):.2f})")

nakrhek → Négatif (confiance 0.81)
nkrahek → Négatif (confiance 0.72)
nekrahek → Négatif (confiance 0.68)
chokrane 3al khedma mte3kom → Positif (confiance 0.78)
nhebbou barcha → Positif (confiance 0.85)


In [2]:
import os
print(os.getcwd())
print(os.path.exists('data/processed/tunizi_clean_binary.csv'))
print(os.path.exists('../data/processed/tunizi_clean_binary.csv'))

C:\Users\jaweher\tounsi-sentiment
True
False


In [55]:
import joblib, os
os.makedirs('models', exist_ok=True)
joblib.dump(model_char, 'models/sentiment_model_char.pkl')
joblib.dump(vec_char, 'models/tfidf_vectorizer_char.pkl')
print("Sauvegardé ✅")

Sauvegardé ✅
